In [ ]:
import tifffile

import numpy as np
from scipy.ndimage import rotate, shift
from sklearn.metrics import mutual_info_score
from skimage.transform import downscale_local_mean


import os
imgfolder = "raw_images"
files = os.listdir(imgfolder)

C1 = {}
C2 = {}
C3 = {}
for i,s in enumerate(['s1','s2','s3']):
    for filename in files:
        print(filename)
        if s in filename:
            if 'before' in filename:
                C1[filename] = tifffile.imread(os.path.join(imgfolder,filename))
            elif 'after' in filename:
                C2[filename] = tifffile.imread(os.path.join(imgfolder,filename))
            elif 'aldehyde' in filename:
                C3[filename] = tifffile.imread(os.path.join(imgfolder,filename))

                

In [2]:
# define functions needed
def mutual_information(img1, img2, bins=64):
    """MI between two images, computed from joint histogram."""
    # Flatten and bin
    i1 = np.clip(img1, 0, None).ravel()
    i2 = np.clip(img2, 0, None).ravel()
    i1 = np.digitize(i1, np.linspace(i1.min(), i1.max(), bins)) - 1
    i2 = np.digitize(i2, np.linspace(i2.min(), i2.max(), bins)) - 1
    return mutual_info_score(i1, i2)


def brute_force_search(fixed, moving, angle_range, dy_range, dx_range, bins=64):
    """Search over all combinations, return best (angle, dy, dx) and MI map."""
    best_mi = -np.inf
    best_params = (0, 0, 0)

    results = {}
    total = len(angle_range) * len(dy_range) * len(dx_range)
    count = 0

    for angle in angle_range:
        rotated = rotate(moving, angle, reshape=False, order=1)
        for dy in dy_range:
            for dx in dx_range:
                shifted = shift(rotated, [dy, dx], order=1)
                mi = mutual_information(fixed, shifted, bins=bins)
                results[(angle, dy, dx)] = mi
                if mi > best_mi:
                    best_mi = mi
                    best_params = (angle, dy, dx)
                count += 1
                if count % 500 == 0:
                    print(f"  {count}/{total}  best MI={best_mi:.4f} @ a={best_params[0]:.1f} dy={best_params[1]:.1f} dx={best_params[2]:.1f}")

    return best_params, best_mi, results


def coarse_to_fine_register(fixed, moving, 
                             angle_range=(-10, 10),
                             shift_range=(-50, 50),
                             n_levels=3,
                             downsample_factor=4):
    """
    Multi-resolution brute force registration.
    
    Parameters
    ----------
    angle_range : tuple
        (min_angle, max_angle) in degrees to search at coarsest level
    shift_range : tuple
        (min_shift, max_shift) in pixels (full-res) to search at coarsest level
    n_levels : int
        Number of coarse-to-fine levels
    downsample_factor : int
        Downsampling per level (cumulative)
    """
    
    # Build pyramid (coarsest first)
    scales = [downsample_factor ** (n_levels - i) for i in range(n_levels)]
    # Last level is scale=1 (full res) if downsample_factor^0 = 1
    # Actually let's keep last level at downsample_factor to stay fast,
    # then do a final fine pass at scale=1
    scales = [downsample_factor ** (n_levels - 1 - i) for i in range(n_levels)]
    
    best_angle, best_dy, best_dx = 0.0, 0.0, 0.0
    
    for level, scale in enumerate(scales):
        print(f"\n--- Level {level} (scale 1/{scale}) ---")
        
        if scale > 1:
            fix_down = downscale_local_mean(fixed, (scale, scale))
            mov_down = downscale_local_mean(moving, (scale, scale))
        else:
            fix_down = fixed.astype(np.float64)
            mov_down = moving.astype(np.float64)
        
        if level == 0:
            # Coarse: wide search
            angles = np.linspace(angle_range[0], angle_range[1], 21)
            shifts_y = np.linspace(shift_range[0] / scale, shift_range[1] / scale, 21)
            shifts_x = np.linspace(shift_range[0] / scale, shift_range[1] / scale, 21)
        else:
            # Refine around previous best (scaled to current resolution)
            prev_scale = scales[level - 1]
            # Scale up shifts from previous level
            best_dy = best_dy * (prev_scale / scale)
            best_dx = best_dx * (prev_scale / scale)
            
            # Narrow search window
            angle_step = (angle_range[1] - angle_range[0]) / (20 * (2 ** level))
            shift_step = max(1.0, 2.0 * prev_scale / scale)
            
            angles = np.linspace(best_angle - 3 * angle_step, best_angle + 3 * angle_step, 7)
            shifts_y = np.linspace(best_dy - 3 * shift_step, best_dy + 3 * shift_step, 7)
            shifts_x = np.linspace(best_dx - 3 * shift_step, best_dx + 3 * shift_step, 7)
        
        n_evals = len(angles) * len(shifts_y) * len(shifts_x)
        print(f"  Searching {len(angles)} angles x {len(shifts_y)} y x {len(shifts_x)} x = {n_evals} evaluations")
        print(f"  Angles: [{angles[0]:.2f}, {angles[-1]:.2f}]")
        print(f"  Shifts Y: [{shifts_y[0]:.1f}, {shifts_y[-1]:.1f}], X: [{shifts_x[0]:.1f}, {shifts_x[-1]:.1f}]")
        
        (best_angle, best_dy, best_dx), best_mi, _ = brute_force_search(
            fix_down, mov_down, angles, shifts_y, shifts_x
        )
        print(f"  Best: angle={best_angle:.3f}°  dy={best_dy:.2f}  dx={best_dx:.2f}  MI={best_mi:.4f}")
    
    # Convert final shifts back to full resolution
    final_scale = scales[-1]
    final_dy = best_dy * final_scale
    final_dx = best_dx * final_scale
    
    print(f"\n=== Final (full-res pixels): angle={best_angle:.3f}°  dy={final_dy:.1f}  dx={final_dx:.1f} ===")
    
    return best_angle, final_dy, final_dx




In [19]:
for i in range(3):

    # Normalise to float
    c1 = C1[list(C1.keys())[i]].astype(np.float32)
    c2 = C2[list(C2.keys())[i]].astype(np.float32)
    c3 = C3[list(C3.keys())[i]].astype(np.float32)

    angle, dy, dx = coarse_to_fine_register(
        c1, c2,
        angle_range=(-5, 5),       # adjust if you expect more rotation
        shift_range=(-100, 100),   # adjust to expected max offset in pixels
        n_levels=3,
        downsample_factor=4,
    )

    # Apply final transform
    C2_reg = shift(rotate(c2, angle, reshape=False, order=3), [dy, dx], order=3)
    tifffile.imwrite(list(C2.keys())[i].replace('.tif','') + "_registered.tif", np.clip(C2_reg,0,65535).astype(C1[list(C1.keys())[i]].dtype))
    print('done ',i)



--- Level 0 (scale 1/16) ---
  Searching 21 angles x 21 y x 21 x = 9261 evaluations
  Angles: [-5.00, 5.00]
  Shifts Y: [-6.2, 6.2], X: [-6.2, 6.2]
  500/9261  best MI=0.1801 @ a=-5.0 dy=-1.9 dx=-4.4
  1000/9261  best MI=0.1801 @ a=-5.0 dy=-1.9 dx=-4.4
  1500/9261  best MI=0.1816 @ a=-4.0 dy=1.2 dx=0.0
  2000/9261  best MI=0.1867 @ a=-3.5 dy=3.8 dx=0.6
  2500/9261  best MI=0.2005 @ a=-2.5 dy=1.9 dx=1.2
  3000/9261  best MI=0.2362 @ a=-2.0 dy=2.5 dx=0.6
  3500/9261  best MI=0.2783 @ a=-1.5 dy=2.5 dx=0.6
  4000/9261  best MI=0.3443 @ a=-1.0 dy=1.9 dx=0.6
  4500/9261  best MI=0.4470 @ a=-0.5 dy=1.9 dx=0.6
  5000/9261  best MI=0.5091 @ a=0.0 dy=1.9 dx=0.6
  5500/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  6000/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  6500/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  7000/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  7500/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  8000/9261  best MI=0.5105 @ a=0.5 dy=1.9 dx=0.6
  8500/9261  best MI=0.5105 @ a=0.5 dy=

In [3]:
for i in range(3):

    # Normalise to float
    c1 = C1[list(C1.keys())[i]].astype(np.float64)

    c3 = C3[list(C3.keys())[i]].astype(np.float64)

    angle, dy, dx = coarse_to_fine_register(
        c1, c3,
        angle_range=(-5, 5),       # adjust if you expect more rotation
        shift_range=(-100, 100),   # adjust to expected max offset in pixels
        n_levels=3,
        downsample_factor=4,
    )

    C3_reg = shift(rotate(c3, angle, reshape=False, order=3), [dy, dx], order=3)
    tifffile.imwrite(list(C3.keys())[i].replace('.tif','') + "_registered.tif", np.clip(C3_reg,0,65535).astype(C1[list(C1.keys())[i]].dtype))
    print('done ',i)


--- Level 0 (scale 1/16) ---
  Searching 21 angles x 21 y x 21 x = 9261 evaluations
  Angles: [-5.00, 5.00]
  Shifts Y: [-6.2, 6.2], X: [-6.2, 6.2]
  500/9261  best MI=0.1146 @ a=-5.0 dy=-0.6 dx=0.6
  1000/9261  best MI=0.1179 @ a=-4.5 dy=-0.6 dx=-1.2
  1500/9261  best MI=0.1179 @ a=-4.5 dy=-0.6 dx=-1.2
  2000/9261  best MI=0.1193 @ a=-3.5 dy=-0.6 dx=-1.2
  2500/9261  best MI=0.1193 @ a=-3.5 dy=-0.6 dx=-1.2
  3000/9261  best MI=0.1193 @ a=-3.5 dy=-0.6 dx=-1.2
  3500/9261  best MI=0.1193 @ a=-3.5 dy=-0.6 dx=-1.2
  4000/9261  best MI=0.1193 @ a=-3.5 dy=-0.6 dx=-1.2
  4500/9261  best MI=0.1281 @ a=-0.5 dy=1.9 dx=-0.6
  5000/9261  best MI=0.1376 @ a=0.0 dy=0.0 dx=0.0
  5500/9261  best MI=0.1683 @ a=0.5 dy=1.9 dx=-1.2
  6000/9261  best MI=0.2110 @ a=1.0 dy=1.9 dx=-1.2
  6500/9261  best MI=0.2752 @ a=2.0 dy=1.9 dx=-1.2
  7000/9261  best MI=0.2752 @ a=2.0 dy=1.9 dx=-1.2
  7500/9261  best MI=0.2752 @ a=2.0 dy=1.9 dx=-1.2
  8000/9261  best MI=0.2752 @ a=2.0 dy=1.9 dx=-1.2
  8500/9261  best MI=